<a href="https://colab.research.google.com/github/sudhars97/Forum-Posts-code/blob/main/DL_M5_RR_C10_Modern_RNN.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import torch
from torch import nn
import yfinance as ticker
from sklearn.preprocessing import StandardScaler
import numpy as np

# 1. DATA ACQUISITION (Same duration as previous chapters)
spy = ticker.download("SPY", start="2004-01-01", end="2026-03-01")
data = spy[['Open', 'High', 'Low', 'Close', 'Volume']].copy()
data['Target'] = data['Close'].shift(-1)
data.dropna(inplace=True)

# Define periods
train_data, test_data = data['2004':'2021'], data['2025':'2026']

scaler = StandardScaler()
train_scaled = scaler.fit_transform(train_data.drop('Target', axis=1))
test_scaled = scaler.transform(test_data.drop('Target', axis=1))

def create_sequences(data, target, window=10):
    X, y = [], []
    for i in range(len(data) - window):
        X.append(data[i:i+window])
        y.append(target.iloc[i+window])
    return torch.tensor(np.array(X), dtype=torch.float32), torch.tensor(np.array(y), dtype=torch.float32)

X_train, y_train = create_sequences(train_scaled, train_data['Target'])
X_test, y_test = create_sequences(test_scaled, test_data['Target'])

# 2. MODERN RNN ARCHITECTURE (LSTM - Section 10.1)
class ModernRNN(nn.Module):
    def __init__(self, input_size=5, hidden_size=64):
        super().__init__()
        # LSTM includes Input, Forget, and Output gates (Section 10.1.1)
        self.lstm = nn.LSTM(input_size, hidden_size, batch_first=True)
        self.fc = nn.Linear(hidden_size, 1)

    def forward(self, x):
        # Gates allow LSTM to decide what to 'forget' vs 'remember'
        out, (h_n, c_n) = self.lstm(x)
        return self.fc(h_n[-1])

# 3. TRAINING & TESTING
model = ModernRNN()
loss_fn, optimizer = nn.MSELoss(), torch.optim.Adam(model.parameters(), lr=0.001)

print("Training Modern RNN (LSTM)...")
for epoch in range(10):
    model.train()
    optimizer.zero_grad()
    loss = loss_fn(model(X_train).squeeze(), y_train)
    loss.backward()
    optimizer.step()
    if (epoch + 1) % 5 == 0:
        print(f"Epoch {epoch+1}: Train Loss {loss.item():.2f}")

model.eval()
with torch.no_grad():
    test_mse = loss_fn(model(X_test).squeeze(), y_test)
    print(f"\nFinal Modern RNN Test MSE (2025-2026): {test_mse.item():.2f}")

/tmp/ipykernel_235/1859927779.py:8: FutureWarning: YF.download() has changed argument auto_adjust default to True
  spy = ticker.download("SPY", start="2004-01-01", end="2026-03-01")
[*********************100%***********************]  1 of 1 completed
/tmp/ipykernel_235/1859927779.py:17: PerformanceWarning: dropping on a non-lexsorted multi-index without a level parameter may impact performance.
  train_scaled = scaler.fit_transform(train_data.drop('Target', axis=1))
/tmp/ipykernel_235/1859927779.py:18: PerformanceWarning: dropping on a non-lexsorted multi-index without a level parameter may impact performance.
  test_scaled = scaler.transform(test_data.drop('Target', axis=1))


Training Modern RNN (LSTM)...
Epoch 5: Train Loss 33644.14
Epoch 10: Train Loss 33607.70

Final Modern RNN Test MSE (2025-2026): 395497.28
